In [0]:
--  read delta table using SQL and create temp view
CREATE OR REPLACE TEMP VIEW customers AS
SELECT *
FROM delta.`/Volumes/data/orders/files/chocolate/customers`;
-- product 
CREATE OR REPLACE TEMP VIEW products AS
SELECT *
FROM delta.`/Volumes/data/orders/files/chocolate/products`;
-- store 
CREATE OR REPLACE TEMP VIEW stores AS
SELECT *
FROM delta.`/Volumes/data/orders/files/chocolate/stores`;
--- sales
CREATE OR REPLACE TEMP VIEW sales AS
SELECT *
FROM delta.`/Volumes/data/orders/files/chocolate/sales`;


--- read from temp view
select * from customers;

-- Find number of customers by gender.
SELECT gender, COUNT(*) AS customer_count
FROM customers
GROUP BY gender;

-- Find total number of orders per customer.
SELECT customer_id, COUNT(*) AS total_orders
FROM sales
GROUP BY customer_id;

-- Get product names along with order IDs (join).
SELECT s.order_id, p.product_name
FROM sales s
JOIN products p
ON s.product_id = p.product_id;


-- Get all loyalty members
SELECT *
FROM customers
WHERE loyalty_member = 1;

-- Get customer details with their orders.
SELECT c.*, s.*
FROM customers c
JOIN sales s
ON c.customer_id = s.customer_id;

-- Combine all tables to create a full order dataset
SELECT
  s.order_id, s.order_date, s.customer_id,c.gender,c.join_date,c.loyalty_member,
  s.product_id,p.product_name,p.category,s.unit_price,
  s.store_id,st.store_name,st.city,s.quantity,s.new_revenue
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
JOIN products p ON s.product_id = p.product_id
JOIN stores st ON s.store_id = st.store_id;

-- Find top 5 most sold products.
SELECT p.product_name, SUM(s.quantity) AS total_sold
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_sold DESC
LIMIT 5;

-- Find total sales per product category.
SELECT p.category, SUM(s.new_revenue) AS total_sales
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.category;

-- Count number of orders per city.
SELECT st.city, COUNT(s.order_id) AS order_count
FROM sales s
JOIN stores st ON s.store_id = st.store_id
GROUP BY st.city;

-- Find most popular product category per store.
WITH t AS (
  SELECT st.store_id,st.store_name,p.category,SUM(s.quantity) AS total_sold,
    ROW_NUMBER() OVER (PARTITION BY st.store_id ORDER BY SUM(s.quantity) DESC) AS rn
  FROM sales s
  JOIN products p ON s.product_id = p.product_id
  JOIN stores st ON s.store_id = st.store_id
  GROUP BY st.store_id, st.store_name, p.category
)
SELECT store_id, store_name, category, total_sold
FROM t
WHERE rn = 1;

-- Rank products based on total quantity sold.
SELECT
  p.product_id, p.product_name, SUM(s.quantity) AS total_quantity_sold,
  RANK() OVER (ORDER BY SUM(s.quantity) DESC) AS product_rank
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_id, p.product_name;

-- Find top product per store using window functions.
WITH ranked_products AS (
  SELECT st.store_id,st.store_name,p.product_id,p.product_name,SUM(s.quantity) AS total_sold,
    ROW_NUMBER() OVER (PARTITION BY st.store_id ORDER BY SUM(s.quantity) DESC) AS rn
  FROM sales s
  JOIN products p ON s.product_id = p.product_id
  JOIN stores st ON s.store_id = st.store_id
  GROUP BY st.store_id, st.store_name, p.product_id, p.product_name
)
SELECT store_id, store_name, product_id, product_name, total_sold
FROM ranked_products
WHERE rn = 1;

-- Assign row numbers to orders per customer based on date.
SELECT
  s.order_id,s.customer_id,s.order_date,
  ROW_NUMBER() OVER (PARTITION BY s.customer_id ORDER BY s.order_date) AS rn
FROM sales s;

-- Find stores with no orders.
SELECT st.store_id, st.store_name, st.city
FROM stores st
LEFT JOIN sales s ON st.store_id = s.store_id
WHERE s.store_id IS NULL;

-- Rank stores based on performance.
SELECT
  st.store_id,
  st.store_name,
  st.city,
  SUM(s.new_revenue) AS total_sales,
  RANK() OVER (ORDER BY SUM(s.new_revenue) DESC) AS store_rank
FROM sales s
JOIN stores st ON s.store_id = st.store_id
GROUP BY st.store_id, st.store_name, st.city;